# R4 v2 · NLI fine-tuning with filtered pairs + hard neutrals (Rathi)

**Runtime → Change runtime type → A100 GPU** (L4 or T4 also work, slower), then **Runtime → Run all**. Run it in one go: the upload cell needs the model trained in the same session.

Uses the same Colab Secrets as R2/R3: `GITHUB_TOKEN`, `HF_TOKEN`, `WANDB_KEY`. Switch on **Notebook access** for all three in the 🔑 panel of *this* notebook first, and click **Grant access** if asked.

Why v2: v1 copied the claim label onto every on-topic snippet and fell below zero-shot on the gold pairs. v2 keeps a weak pair only if the zero-shot model agrees, and adds "hard neutral" pairs (a claim + its own snippet that does not decide it).

Steps: 1 setup → 2 tests + 10-row checks → 3 weak pairs → 4 fine-tune mDeBERTa → 5 score every file → 6 upload model → 7 push → 8 XLM-R-large comparison on v2 pairs (optional, ~30 min) → 9 push comparison.

In [ ]:
# 1. Setup. The repo is public, so cloning needs no token; GITHUB_TOKEN is only used to push.
from google.colab import userdata
import os, json
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q rathi/R4-nli && git pull -q origin rathi/R4-nli

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
ok = !git ls-remote https://{GH}@github.com/himaallu/Group-Project.git HEAD
print('GitHub token OK' if ok and 'fatal' not in ' '.join(ok) else 'GITHUB_TOKEN rejected: make a new one with the "repo" box ticked')

!pip install -q -U wandb
import wandb
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
wandb.login(key=userdata.get('WANDB_KEY').strip(), relogin=True)
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
HUB_ID = f"{api.whoami()['name']}/twincheck-nli-v2"
print('model will go to', HUB_ID)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Tests + 10-row checks (pairs, a tiny training run, scoring)

In [ ]:
!python -m pytest tests/nli -q
ZS = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'   # zero-shot model used as the filter
!python -m src.nli.build_pairs --filter-model {ZS} --n-per-class 10 --n-val-per-class 10 --out data/nli/test10/train.jsonl     --val-out data/nli/test10/val_weak.jsonl --report data/nli/test10/report.json
!head -n 3 data/nli/test10/train.jsonl
!python -m src.nli.train --train data/nli/test10/train.jsonl --val data/nli/test10/val_weak.jsonl     --n-train 30 --version test10 --out /content/test10_nli --metrics /content/test10_eval.json     --ckpt-dir /content/ckpt_test10 --no-wandb
!python -m src.nli.score data/processed/en/multifc_val.jsonl --limit 10     --model /content/test10_nli --out-dir /content/test10_scores
import pandas as pd
print(pd.read_parquet('/content/test10_scores/multifc_val_first10.parquet').head())

## 3. v2 training pairs: zero-shot agreement filter + hard neutrals (~5-10 min of scoring)

In [ ]:
!python -m src.nli.build_pairs --filter-model {ZS}
print(open('results/R4/pairs_report.json').read())

## 4. Fine-tune mDeBERTa-v3 on the v2 pairs (2 epochs, lr 2e-5, batch 16, fp32/TF32, seed 42; ~10-20 min)

In [ ]:
!python -m src.nli.train --version v2 --out models/nli_v2
ev = json.load(open('results/R4/gold_eval.json'))
for when in ('zero_shot', 'fine_tuned'):
    print(when, {k: v['macro_f1'] for k, v in ev[when].items()})
gold = lambda when: (ev[when]['gold_en']['macro_f1'] + ev[when]['gold_ar']['macro_f1']) / 2
print('mean gold macro-F1: zero-shot', round(gold('zero_shot'), 4), '| v2', round(gold('fine_tuned'), 4))
print('W&B:', ev['wandb_url'])

## 5. Score every (claim, snippet) pair with the v2 model → `features/nli_v2/`

In [ ]:
FILES = ' '.join([f'data/processed/en/{d}_{s}.jsonl' for d in ('multifc', 'averitec') for s in ('train', 'val', 'test')]
                 + ['data/copies/en/test.jsonl', 'data/copies/en/train.jsonl']
                 + [f'data/processed/ar/xfact_ar_{s}.jsonl' for s in ('train', 'val', 'test')]
                 + ['data/processed/ar/m4fc_ar_test.jsonl'])
!python -m src.nli.score {FILES} --model models/nli_v2 --out-dir features/nli_v2
!du -h features/nli_v2/*

## 6. Upload the v2 model to Hugging Face Hub (too big for git) and write a pointer README

In [ ]:
api.create_repo(HUB_ID, exist_ok=True, private=False)
api.upload_folder(folder_path='models/nli_v2', repo_id=HUB_ID,
                  commit_message='R4 v2: mDeBERTa-v3 XNLI fine-tuned on filtered TwinCheck NLI pairs')
f1 = lambda when, key: ev[when].get(key, {}).get('macro_f1')
lines = ['# nli_v2 (R4)', '',
         f'Weights are on Hugging Face Hub: https://huggingface.co/{HUB_ID} (too big for git).', '',
         f'Load: `python -m src.nli.score FILE --model {HUB_ID}`.', '',
         f"Base `{ev['base_model']}`, {ev['epochs']} epochs, {ev['n_train_pairs']} v2 pairs, lr {ev['lr']}, "
         f"seed {ev['seed']}, {ev['precision']}. Gold macro-F1 EN {f1('zero_shot', 'gold_en')} -> "
         f"{f1('fine_tuned', 'gold_en')}, AR {f1('zero_shot', 'gold_ar')} -> {f1('fine_tuned', 'gold_ar')}. "
         f"W&B: {ev['wandb_url']}"]
open('models/nli_v2/README.md', 'w').write(chr(10).join(lines) + chr(10))
print(open('models/nli_v2/README.md').read())

## 7. Push the outputs to `rathi/R4-nli`

In [ ]:
!git add data/nli/train.jsonl data/nli/val_weak.jsonl features/nli_v2 models/nli_v2/README.md     results/R4/pairs_report.json results/R4/gold_eval.json
!git commit -q -m "[R4] Add v2 NLI pairs, v2 scores and gold eval from Colab run"
!git push -q origin rathi/R4-nli && echo PUSHED

## 8. XLM-R-large comparison (optional, ~40 min on an A100). Same data and settings, lr 1e-5 (large models are unstable at 2e-5). Only the gold numbers are kept.

In [ ]:
!python -m src.nli.train --model joeddav/xlm-roberta-large-xnli --lr 1e-5 --version v2     --out /content/nli_xlmr --metrics results/R4/gold_eval_xlmr.json --ckpt-dir /content/ckpt_xlmr
xl = json.load(open('results/R4/gold_eval_xlmr.json'))
for when in ('zero_shot', 'fine_tuned'):
    print(when, {k: v['macro_f1'] for k, v in xl[when].items()})

## 9. Push the comparison

In [ ]:
!git add results/R4/gold_eval_xlmr.json
!git commit -q -m "[R4] Add XLM-R-large comparison on the v2 pairs"
!git push -q origin rathi/R4-nli && echo PUSHED

**Done.** Tell Claude it finished. Then **Runtime → Disconnect and delete runtime**.